# Sports Watcher: M1 vision weights

Trains the two RF-DETR models on a Colab GPU and hands back a `models.zip` for the Mac.

Before running: **Runtime → Change runtime type → GPU (T4)**. Then, in the left sidebar,
open **Secrets** (key icon), add `ROBOFLOW_API_KEY` with your Roboflow private key and
allow notebook access. Finally upload `training-colab.zip` (from the repo's `tmp/`
folder, made with `zip -r tmp/training-colab.zip training -x "*/__pycache__/*"`) through
the **Files** panel.

Run the cells top to bottom. Expect one to two hours per model on a T4; keep the tab open.


In [ ]:
!nvidia-smi -L
!pip -q install "rfdetr[train,onnx]" roboflow


In [ ]:
import os
from pathlib import Path

from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
assert Path("training-colab.zip").exists(), "upload training-colab.zip through the Files panel first"
!unzip -q -o training-colab.zip
print(sorted(p.name for p in Path("training").glob("*.py")))


In [ ]:
from training.datasets import download_roboflow, remap_split_dirs

PLAYERS_VERSION = 20  # the latest on Roboflow Universe at the time of writing
download_roboflow("roboflow-jvuqo", "basketball-player-detection-2", PLAYERS_VERSION, "coco", Path("data/datasets/players"))
print(remap_split_dirs(Path("data/datasets/players")))
download_roboflow("roboflow-jvuqo", "basketball-court-detection-2", 19, "coco", Path("data/datasets/court"))


## Detector (ball, player, referee, rim)

Small at 512 is the plan's default. For the size comparison in the notes, run the second
cell too (Nano at 384) when time allows; it lands under `models/detector-nano`.

Batch 4 with `--grad-accum 4` (effective 16) is what fits a 16 GB T4: batch 8 ran out of
GPU memory on the court model, which trains at 696 px. The Colab VM has two CPUs, hence
`--workers 2`.

In [ ]:
!python -m training.train_detector --dataset-dir data/datasets/players --dataset-version 20 \
    --model small --resolution 512 --epochs 30 --batch 4 --grad-accum 4 --workers 2 --device cuda \
    --note "small 512, 30 epochs, Colab T4" --out models/detector

In [ ]:
# optional second size for the benchmark
!python -m training.train_detector --dataset-dir data/datasets/players --dataset-version 20 \
    --model nano --resolution 384 --epochs 30 --batch 8 --grad-accum 2 --workers 2 --device cuda \
    --note "nano 384, 30 epochs, Colab T4" --out models/detector-nano

## Court keypoints (33 points)

In [ ]:
!python -m training.train_court --dataset-dir data/datasets/court --resolution 576 --epochs 40 \
    --batch 4 --grad-accum 4 --workers 2 --device cuda --out models/court

## Package and download

The zip leaves the `run/` folders (checkpoints) out. On the Mac:

    ./.venv/bin/python scripts/download_models.py --zip ~/Downloads/models.zip


In [ ]:
!rm -f models.zip && zip -r -q models.zip models -x "models/*/run/*"
!ls -la models.zip && unzip -l models.zip
from google.colab import files

files.download("models.zip")
